# Agente RAG Corporativo — Azure OpenAI + Chroma (em memória)

Assistente que responde perguntas sobre documentos internos **com citações**, seguindo o roteiro de 5 passos:

1. **Preparar os documentos** — coletar e fazer chunking (500–1000 tokens, overlap 50–100)
2. **Gerar embeddings** — `text-embedding-3-small` → vector store (**Chroma em memória**; Azure AI Search em produção)
3. **Montar a chain RAG** — retriever (`top_k=5`) + prompt com `{context}` e `{question}`, resposta com citações
4. **Adicionar tools ao agente** — busca na KB + consulta a APIs externas (Jira, HubSpot mockados)
5. **Avaliar e iterar** — Q&A de referência, recall do retriever e qualidade da geração

> Tudo roda **em memória** e com documentos embutidos — não precisa subir nada para testar. Só as credenciais do Azure.

## 0. Requisitos

Instale as dependências (apenas na primeira vez).

In [ ]:
%pip install openai langchain langchain-openai langchain-chroma langchain-text-splitters langgraph chromadb tiktoken python-dotenv --quiet

## 0.1 Credenciais

Crie um arquivo `.env` na mesma pasta:

```env
AZURE_OPENAI_ENDPOINT=https://SEU-RECURSO.openai.azure.com/
AZURE_OPENAI_KEY=sua-chave-aqui
```

Os valores de `azure_deployment` abaixo são os **nomes dos deployments** no seu recurso — ajuste se necessário.

In [ ]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
KEY = os.getenv("AZURE_OPENAI_KEY")
API_VERSION = "2024-10-21"  # GA, suporta chat, tools e embeddings

assert ENDPOINT and KEY, "Configure AZURE_OPENAI_ENDPOINT e AZURE_OPENAI_KEY no .env"

# Modelo de chat (usado na chain RAG e no agente)
llm = AzureChatOpenAI(
    azure_endpoint=ENDPOINT,
    api_key=KEY,
    api_version=API_VERSION,
    azure_deployment="gpt-4o",   # nome do deployment de chat no Azure
    temperature=0,
)

# Modelo de embeddings
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=ENDPOINT,
    api_key=KEY,
    api_version=API_VERSION,
    azure_deployment="text-embedding-3-small",  # nome do deployment de embedding
)

print("Azure OpenAI configurado.")

## Passo 1 — Preparar os documentos

Numa demo usamos documentos internos **embutidos** (política de home office, reembolso, segurança, benefícios e onboarding). Em produção você trocaria isso por loaders de PDF/DOCX/wiki.

O chunking usa contagem por **tokens** (`from_tiktoken_encoder`) com `chunk_size=600` e `chunk_overlap=80`, dentro da faixa recomendada.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Base de conhecimento simulada (documentos internos) ---
documentos = [
    Document(
        page_content=(
            "Política de Home Office. "
            "Colaboradores podem trabalhar remotamente até 3 dias por semana, mediante acordo com o gestor. "
            "Os dias presenciais obrigatorios sao terca e quinta. "
            "Para home office integral e necessaria aprovacao do diretor da area e registro no RH. "
            "A empresa fornece auxilio home office de R$ 150 mensais para custeio de internet e energia. "
            "Reunioes de equipe devem priorizar horarios entre 9h e 17h no fuso de Brasilia."
        ),
        metadata={"source": "politica_home_office.md", "categoria": "RH"},
    ),
    Document(
        page_content=(
            "Política de Reembolso de Despesas. "
            "Despesas de viagem a trabalho sao reembolsaveis mediante nota fiscal e aprovacao do gestor. "
            "O limite diario de alimentacao em viagem e de R$ 120. "
            "Transporte por aplicativo e reembolsavel apenas para deslocamentos a trabalho, nao para o trajeto casa-trabalho. "
            "Solicitacoes de reembolso devem ser enviadas em ate 30 dias apos a despesa, pelo sistema Concur. "
            "Reembolsos aprovados sao pagos junto com o salario do mes seguinte."
        ),
        metadata={"source": "politica_reembolso.md", "categoria": "Financeiro"},
    ),
    Document(
        page_content=(
            "Guia de Seguranca da Informacao. "
            "Toda senha corporativa deve ter no minimo 12 caracteres e ser trocada a cada 90 dias. "
            "O uso de gerenciador de senhas corporativo e obrigatorio. "
            "E proibido compartilhar credenciais entre colaboradores, mesmo dentro da mesma equipe. "
            "Dados classificados como confidenciais nao podem ser enviados por e-mail pessoal nem armazenados em nuvem nao homologada. "
            "Todo incidente de seguranca deve ser reportado ao time de SecOps em ate 1 hora pelo canal #seguranca no Slack. "
            "Notebooks corporativos possuem criptografia de disco obrigatoria e bloqueio automatico apos 5 minutos de inatividade. "
            "O acesso a sistemas criticos exige autenticacao multifator (MFA) sem excecao. "
            "Colaboradores devem concluir o treinamento anual de seguranca ate 31 de marco."
        ),
        metadata={"source": "seguranca_informacao.md", "categoria": "Seguranca"},
    ),
    Document(
        page_content=(
            "FAQ de Beneficios. "
            "O plano de saude cobre o colaborador e dependentes diretos, com coparticipacao de 20% em consultas. "
            "O vale-refeicao e de R$ 1.200 mensais, creditados no dia 1 de cada mes. "
            "A empresa oferece Gympass no plano basico sem custo e planos superiores com desconto em folha. "
            "Ha licenca-paternidade estendida de 20 dias e licenca-maternidade de 6 meses. "
            "O programa de participacao nos lucros (PLR) e pago em duas parcelas, em fevereiro e agosto."
        ),
        metadata={"source": "beneficios_faq.md", "categoria": "RH"},
    ),
    Document(
        page_content=(
            "Procedimento de Onboarding. "
            "Novos colaboradores recebem notebook e acessos no primeiro dia, provisionados pelo time de TI. "
            "A trilha de integracao dura 5 dias uteis e inclui apresentacao das areas e treinamentos obrigatorios. "
            "O buddy designado acompanha o novo colaborador nas primeiras duas semanas. "
            "A avaliacao de experiencia (periodo de 90 dias) tem checkpoints em 30, 60 e 90 dias. "
            "Duvidas de onboarding devem ser direcionadas ao canal #pessoas no Slack."
        ),
        metadata={"source": "onboarding.md", "categoria": "RH"},
    ),
]

# Chunking por tokens (600 tokens, overlap 80)
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=600,
    chunk_overlap=80,
)
chunks = splitter.split_documents(documentos)

print(f"{len(documentos)} documentos -> {len(chunks)} chunks")
for c in chunks[:2]:
    print("-", c.metadata["source"], "|", c.page_content[:60], "...")

## Passo 2 — Gerar embeddings e indexar (Chroma em memória)

Sem `persist_directory`, o Chroma roda **em memória** (efêmero) — perfeito para testes/demos. Em produção você trocaria por Azure AI Search mantendo praticamente a mesma interface de `retriever`.

In [ ]:
from langchain_chroma import Chroma

# Chroma em memória (efêmero). Nome da collection precisa ter 3+ caracteres.
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="base_conhecimento",
)

# Retriever com top_k=5
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# Teste rápido do retriever
_hits = retriever.invoke("quantos dias de home office posso fazer?")
print(f"{len(_hits)} chunks recuperados:")
for d in _hits:
    print(" -", d.metadata["source"])

## Passo 3 — Montar a chain RAG (com citações)

Prompt com placeholders `{context}` e `{question}`. Os trechos recuperados são **numerados** e o modelo é instruído a citar as fontes usando `[n]`. Se a resposta não estiver no contexto, ele deve dizer que não sabe.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

SYSTEM = (
    "Voce e um assistente corporativo. Responda a pergunta do usuario "
    "APENAS com base nos trechos de contexto fornecidos. "
    "Cite as fontes usando os numeros entre colchetes correspondentes, ex: [1], [2]. "
    "Se a resposta nao estiver no contexto, diga claramente que nao encontrou a informacao "
    "nos documentos internos. Responda em portugues, de forma objetiva."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "Contexto:\n{context}\n\nPergunta: {question}"),
])

def format_docs(docs):
    """Numera os trechos e anexa a fonte, para permitir citacao [n]."""
    linhas = []
    for i, d in enumerate(docs, start=1):
        linhas.append(f"[{i}] (fonte: {d.metadata['source']})\n{d.page_content}")
    return "\n\n".join(linhas)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def perguntar(pergunta: str):
    """Executa a chain e tambem mostra o mapa de fontes [n] -> arquivo."""
    docs = retriever.invoke(pergunta)
    resposta = rag_chain.invoke(pergunta)
    print("PERGUNTA:", pergunta)
    print("\nRESPOSTA:\n", resposta)
    print("\nFONTES:")
    for i, d in enumerate(docs, start=1):
        print(f"  [{i}] {d.metadata['source']}")
    return resposta

In [ ]:
# Teste da chain RAG
_ = perguntar("Quantos dias por semana posso trabalhar de casa e quais dias sao presenciais?")

In [ ]:
# Pergunta cuja resposta NAO esta nos documentos (deve admitir que nao sabe)
_ = perguntar("Qual e a politica de estacionamento no escritorio?")

## Passo 4 — Adicionar tools ao agente

Agora envolvemos o RAG numa **tool** e adicionamos **APIs externas mockadas** (Jira e HubSpot). O agente (ReAct via LangGraph) decide sozinho qual ferramenta usar em cada pergunta.

> As funções de Jira/HubSpot retornam dados fake — troque pela integração real quando for para produção.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def buscar_base_conhecimento(pergunta: str) -> str:
    """Busca informacoes nos documentos internos da empresa (RH, financeiro, seguranca,
    beneficios, onboarding). Use para politicas e procedimentos internos."""
    docs = retriever.invoke(pergunta)
    return format_docs(docs)

# --- Mock de API externa: Jira ---
_JIRA = {
    "PROJ-101": {"status": "Em andamento", "responsavel": "Ana Souza", "titulo": "Migracao do vector store para Azure AI Search"},
    "PROJ-102": {"status": "Concluido", "responsavel": "Carlos Lima", "titulo": "Pipeline de ingestao de PDFs"},
}

@tool
def consultar_jira(chave_ticket: str) -> str:
    """Consulta o status de um ticket no Jira pela chave (ex: PROJ-101).
    Use quando o usuario perguntar sobre andamento de tarefas ou tickets."""
    t = _JIRA.get(chave_ticket.upper())
    if not t:
        return f"Ticket {chave_ticket} nao encontrado."
    return f"{chave_ticket}: {t['titulo']} | status={t['status']} | responsavel={t['responsavel']}"

# --- Mock de API externa: HubSpot ---
_HUBSPOT = {
    "maria@cliente.com": {"empresa": "Cliente XPTO", "estagio": "Proposta enviada", "owner": "Joao Pereira"},
}

@tool
def consultar_hubspot(email: str) -> str:
    """Consulta um contato/negocio no HubSpot pelo e-mail.
    Use quando o usuario perguntar sobre clientes, leads ou oportunidades de venda."""
    c = _HUBSPOT.get(email.lower())
    if not c:
        return f"Contato {email} nao encontrado no CRM."
    return f"{email} | empresa={c['empresa']} | estagio={c['estagio']} | owner={c['owner']}"

tools = [buscar_base_conhecimento, consultar_jira, consultar_hubspot]

AGENT_PROMPT = (
    "Voce e o assistente corporativo da empresa. "
    "Use buscar_base_conhecimento para politicas e procedimentos internos. "
    "Use consultar_jira para status de tickets e consultar_hubspot para clientes/CRM. "
    "Sempre que usar a base de conhecimento, cite as fontes. Responda em portugues."
)

agente = create_react_agent(llm, tools, prompt=AGENT_PROMPT)
print("Agente criado com", len(tools), "tools.")

In [ ]:
def rodar_agente(pergunta: str):
    resultado = agente.invoke({"messages": [{"role": "user", "content": pergunta}]})
    # ultima mensagem = resposta final
    print("PERGUNTA:", pergunta)
    print("RESPOSTA:", resultado["messages"][-1].content)
    print("-" * 70)

# Pergunta que cai na base de conhecimento
rodar_agente("Qual o limite diario de alimentacao em viagem?")

# Pergunta que cai numa API externa (Jira)
rodar_agente("Qual o status do ticket PROJ-101?")

# Pergunta que cai no CRM (HubSpot)
rodar_agente("Em que estagio esta o negocio do contato maria@cliente.com?")

## Passo 5 — Avaliar e iterar

Criamos um conjunto de Q&A de referência. Numa demo usamos ~6 perguntas; **em produção você criaria 20–30**. Medimos:

- **Recall@k do retriever** — a fonte esperada apareceu entre os `k` chunks recuperados?
- **Qualidade da geração** — a resposta contém a informação esperada (checagem simples por palavra-chave).

Depois é iterar: ajustar `chunk_size`/`overlap`, `top_k` e o prompt.

In [ ]:
# Conjunto de referencia (expanda para 20-30 em producao)
referencia = [
    {"pergunta": "Quantos dias de home office por semana sao permitidos?",
     "fonte_esperada": "politica_home_office.md", "esperado_contem": "3"},
    {"pergunta": "Qual o limite diario de alimentacao em viagem?",
     "fonte_esperada": "politica_reembolso.md", "esperado_contem": "120"},
    {"pergunta": "De quanto em quanto tempo a senha deve ser trocada?",
     "fonte_esperada": "seguranca_informacao.md", "esperado_contem": "90"},
    {"pergunta": "Qual o valor do vale-refeicao?",
     "fonte_esperada": "beneficios_faq.md", "esperado_contem": "1.200"},
    {"pergunta": "Quanto tempo dura a trilha de onboarding?",
     "fonte_esperada": "onboarding.md", "esperado_contem": "5"},
    {"pergunta": "Qual o prazo para enviar solicitacao de reembolso?",
     "fonte_esperada": "politica_reembolso.md", "esperado_contem": "30"},
]

# --- Recall@k do retriever ---
def recall_retriever(dataset, k=5):
    acertos = 0
    for item in dataset:
        docs = vector_store.as_retriever(search_kwargs={"k": k}).invoke(item["pergunta"])
        fontes = {d.metadata["source"] for d in docs}
        ok = item["fonte_esperada"] in fontes
        acertos += ok
        print(("OK " if ok else "XX "), item["pergunta"][:45], "->", sorted(fontes))
    print(f"\nRecall@{k}: {acertos}/{len(dataset)} = {acertos/len(dataset):.0%}")

recall_retriever(referencia, k=5)

In [ ]:
# --- Qualidade da geracao (checagem simples por palavra-chave) ---
def avaliar_geracao(dataset):
    acertos = 0
    for item in dataset:
        resposta = rag_chain.invoke(item["pergunta"])
        ok = item["esperado_contem"].lower() in resposta.lower()
        acertos += ok
        print(("OK " if ok else "XX "), item["pergunta"][:45], "| esperado:", item["esperado_contem"])
    print(f"\nAcerto de geracao: {acertos}/{len(dataset)} = {acertos/len(dataset):.0%}")

avaliar_geracao(referencia)

### Como iterar a partir daqui

- **Recall baixo?** Reduza `chunk_size`, aumente `overlap`, suba o `top_k`, ou melhore os metadados/limpeza dos documentos.
- **Geração ruim mesmo com bom recall?** Refine o prompt (instruções de citação, tom, "não invente"), ou teste re-ranking dos trechos.
- **LLM-as-judge:** para avaliação mais rica que palavra-chave, use o próprio `llm` como juiz, pontuando fidelidade ao contexto e completude de 1 a 5.
- **Produção:** troque Chroma por **Azure AI Search**, adicione cache de embeddings, e registre métricas por versão de prompt/chunking para comparar iterações.